**Phase 3: Metrics**

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

df = pd.read_csv("../data/processed/marketing_clean.csv", parse_dates=["date", "month"])
RAW_COLS = ["impressions", "clicks", "ad_spend", "conversions", "revenue"]


def safe_div(a, b):
    return a.div(b.where(b != 0))


def add_metrics(g):
    g = g.copy()
    g["CTR"] = safe_div(g["clicks"], g["impressions"])
    g["CPC"] = safe_div(g["ad_spend"], g["clicks"])
    g["conv_rate"] = safe_div(g["conversions"], g["clicks"])
    g["CAC"] = safe_div(g["ad_spend"], g["conversions"])
    g["ROAS"] = safe_div(g["revenue"], g["ad_spend"])
    return g


def summarize(by):
    g = add_metrics(df.groupby(by)[RAW_COLS].sum())
    g["spend_share"] = g["ad_spend"] / g["ad_spend"].sum()
    g["revenue_share"] = g["revenue"] / g["revenue"].sum()
    return g.sort_values("ROAS", ascending=False)

In [2]:
by_platform = summarize("platform")
by_platform

,impressions,clicks,ad_spend,conversions,revenue,CTR,CPC,conv_rate,CAC,ROAS,spend_share,revenue_share
platform,,,,,,,,,,,,
TikTok Ads,47049590,2599931,"2,653,418.5100",122452,"20,223,540.0700",0.0553,1.0206,0.0471,21.6690,7.6217,0.2389,0.3732
Meta Ads,64403494,1596736,"2,106,061.6700",73262,"11,926,045.7900",0.0248,1.3190,0.0459,28.7470,5.6627,0.1896,0.2201
Google Ads,73801150,2936149,"6,349,268.9100",131098,"22,033,744.9500",0.0398,2.1624,0.0446,48.4315,3.4703,0.5716,0.4067


In [3]:
row_roas = safe_div(df["revenue"], df["ad_spend"])
trimmed = df.drop(row_roas.nlargest(5).index)

by_platform["ROAS_no_top5"] = add_metrics(
    trimmed.groupby("platform")[RAW_COLS].sum()
)["ROAS"]
by_platform[["ROAS", "ROAS_no_top5"]]

,ROAS,ROAS_no_top5
platform,,
TikTok Ads,7.6217,7.4669
Meta Ads,5.6627,5.6627
Google Ads,3.4703,3.4703


In [4]:
by_type = summarize("campaign_type")
by_type

,impressions,clicks,ad_spend,conversions,revenue,CTR,CPC,conv_rate,CAC,ROAS,spend_share,revenue_share
campaign_type,,,,,,,,,,,,
Search,47544475,1898227,"2,868,006.8500",90641,"15,218,470.8500",0.0399,1.5109,0.0478,31.6414,5.3063,0.2582,0.2809
Display,43487104,1638396,"2,644,735.1200",76869,"12,798,903.1700",0.0377,1.6142,0.0469,34.4057,4.8394,0.2381,0.2362
Video,47748617,1810883,"2,796,458.4800",79228,"13,341,261.1900",0.0379,1.5443,0.0438,35.2963,4.7708,0.2517,0.2462
Shopping,46474038,1785310,"2,799,548.6400",80074,"12,824,695.6000",0.0384,1.5681,0.0449,34.9620,4.5810,0.2520,0.2367


In [5]:
pt = add_metrics(df.groupby(["platform", "campaign_type"])[RAW_COLS].sum())

print("ROAS (platform x campaign_type)")
display(pt["ROAS"].unstack("campaign_type").round(2))
print("CAC (platform x campaign_type)")
display(pt["CAC"].unstack("campaign_type").round(2))
print("Row counts")
display(df.groupby(["platform", "campaign_type"]).size().unstack())

ROAS (platform x campaign_type)


campaign_type,Display,Search,Shopping,Video
platform,,,,
Google Ads,3.1600,3.9200,3.2300,3.5300
Meta Ads,5.8200,5.7700,5.5400,5.5100
TikTok Ads,8.0100,8.1200,6.8500,7.5600


CAC (platform x campaign_type)


campaign_type,Display,Search,Shopping,Video
platform,,,,
Google Ads,50.5500,44.2500,51.7600,48.1600
Meta Ads,29.3900,28.4100,26.2400,31.2100
TikTok Ads,21.2900,20.2300,23.6900,21.6900


Row counts


campaign_type,Display,Search,Shopping,Video
platform,,,,
Google Ads,174,192,168,186
Meta Ads,145,160,160,165
TikTok Ads,101,125,119,105


In [6]:
print("Lowest CAC platform:", by_platform["CAC"].idxmin(), round(by_platform["CAC"].min(), 2))
print("Highest ROAS platform:", by_platform["ROAS"].idxmax(), round(by_platform["ROAS"].max(), 2))
print("Highest conv_rate campaign type:", by_type["conv_rate"].idxmax(), round(by_type["conv_rate"].max(), 4))
print("Highest ROAS campaign type:", by_type["ROAS"].idxmax(), round(by_type["ROAS"].max(), 2))

Lowest CAC platform: TikTok Ads 21.67
Highest ROAS platform: TikTok Ads 7.62
Highest conv_rate campaign type: Search 0.0478
Highest ROAS campaign type: Search 5.31


- Platform ROAS: TikTok 7.62, Meta 5.66, Google 3.47 | CAC: 21.67 / 28.75 / 48.43
- Google: 57% of spend, 41% of revenue. TikTok: 24% of spend, 37% of revenue
- TikTok leads in every campaign type; robust to top-5 outlier removal (7.62 -> 7.47)
- ROAS gap comes from CAC/CPC, not conversion rate (4.5-4.7% on all platforms)
- Campaign type: Search best (ROAS 5.31, conv. rate 4.78%), spread is small (1.16x vs 2.2x for platforms)

In [7]:
COLS = ["ad_spend", "CTR", "CPC", "conv_rate", "CAC", "ROAS", "spend_share", "revenue_share"]

by_industry = summarize("industry")
by_country = summarize("country")

display(by_industry[COLS])
display(by_country[COLS])

,ad_spend,CTR,CPC,conv_rate,CAC,ROAS,spend_share,revenue_share
industry,,,,,,,,
SaaS,"2,357,561.8400",0.0408,1.5200,0.0459,33.0835,5.0442,0.2122,0.2195
EdTech,"2,296,978.5200",0.0377,1.5291,0.0448,34.1523,5.0283,0.2068,0.2132
E-commerce,"1,923,987.0800",0.0379,1.5030,0.0449,33.4688,5.0090,0.1732,0.1779
Healthcare,"2,258,683.9200",0.0379,1.6283,0.0474,34.3856,4.8395,0.2033,0.2017
Fintech,"2,271,537.7300",0.0382,1.6083,0.0461,34.8818,4.4786,0.2045,0.1878


,ad_spend,CTR,CPC,conv_rate,CAC,ROAS,spend_share,revenue_share
country,,,,,,,,
India,"1,438,741.7600",0.0383,1.4164,0.0460,30.7640,5.4976,0.1295,0.1460
Australia,"1,586,264.9400",0.0404,1.6508,0.0477,34.5825,4.9940,0.1428,0.1462
Germany,"1,551,062.1700",0.0381,1.5595,0.0450,34.6467,4.9610,0.1396,0.1420
UK,"1,547,153.4800",0.0374,1.5369,0.0465,33.0292,4.9507,0.1393,0.1414
UAE,"1,614,677.3300",0.0394,1.5287,0.0458,33.3991,4.9171,0.1454,0.1465
Canada,"1,643,262.5400",0.0378,1.5880,0.0447,35.5354,4.7876,0.1479,0.1452
USA,"1,727,586.8700",0.0385,1.6239,0.0451,36.0079,4.1622,0.1555,0.1327


In [8]:
pi = add_metrics(df.groupby(["platform", "industry"])[RAW_COLS].sum())
print("ROAS (platform x industry)")
display(pi["ROAS"].unstack("industry").round(2))
print("Row counts")
display(df.groupby(["platform", "industry"]).size().unstack())

pc = add_metrics(df.groupby(["platform", "country"])[RAW_COLS].sum())
print("ROAS (platform x country)")
display(pc["ROAS"].unstack("country").round(2))
print("Row counts")
display(df.groupby(["platform", "country"]).size().unstack())

ROAS (platform x industry)


industry,E-commerce,EdTech,Fintech,Healthcare,SaaS
platform,,,,,
Google Ads,3.7000,3.5800,3.7000,3.1200,3.2900
Meta Ads,4.9700,6.2200,4.8300,6.5100,5.7000
TikTok Ads,8.3600,7.3500,6.3800,7.6800,8.2600


Row counts


industry,E-commerce,EdTech,Fintech,Healthcare,SaaS
platform,,,,,
Google Ads,133,143,162,137,145
Meta Ads,139,135,118,127,111
TikTok Ads,77,94,81,84,114


ROAS (platform x country)


country,Australia,Canada,Germany,India,UAE,UK,USA
platform,,,,,,,
Google Ads,3.4900,3.1500,3.4900,3.8200,3.8000,3.7000,2.9700
Meta Ads,6.5200,5.2700,6.5600,5.8500,6.0300,5.1200,4.8800
TikTok Ads,8.7500,8.5200,7.1000,7.8000,7.0600,7.6100,6.5300


Row counts


country,Australia,Canada,Germany,India,UAE,UK,USA
platform,,,,,,,
Google Ads,103,103,94,96,121,97,106
Meta Ads,74,87,96,89,76,108,100
TikTok Ads,55,72,65,76,61,61,60


In [9]:
monthly = add_metrics(df.groupby("month")[RAW_COLS].sum())
display(monthly[["ad_spend", "revenue", "CAC", "ROAS"]])

monthly_platform = add_metrics(df.groupby(["month", "platform"])[RAW_COLS].sum())
display(monthly_platform["ROAS"].unstack("platform").round(2))

,ad_spend,revenue,CAC,ROAS
month,,,,
2024-01-01,"901,735.0600","4,691,994.4700",32.1531,5.2033
2024-02-01,"988,588.3600","4,749,203.4600",34.6108,4.8040
2024-03-01,"837,915.2800","4,407,519.3800",34.0229,5.2601
2024-04-01,"1,032,895.9000","5,143,463.5700",35.0253,4.9797
2024-05-01,"904,564.2600","3,768,913.3600",36.1291,4.1666
2024-06-01,"966,952.7600","4,362,176.3100",34.2806,4.5113
2024-07-01,"797,918.1400","3,695,093.9700",36.4014,4.6309
2024-08-01,"819,622.0500","4,301,659.2600",31.0287,5.2483
2024-09-01,"1,010,523.0400","4,749,430.2700",34.2052,4.7000


platform,Google Ads,Meta Ads,TikTok Ads
month,,,
2024-01-01,3.4500,6.8200,6.6100
2024-02-01,3.4500,4.9000,7.6100
2024-03-01,3.8000,6.2600,9.0300
2024-04-01,3.6000,5.9100,7.5200
2024-05-01,2.8100,5.0800,7.9900
2024-06-01,3.4900,5.6400,6.6600
2024-07-01,3.1900,5.1700,8.0900
2024-08-01,4.1700,5.5200,7.6500
2024-09-01,3.3300,4.5700,8.0100


In [10]:
OUT_DIR = "../data/processed/"
tables = {
    "platform": by_platform,
    "campaign_type": by_type,
    "industry": by_industry,
    "country": by_country,
    "platform_campaign": pt,
    "monthly_platform": monthly_platform,
}
for name, t in tables.items():
    out = t.reset_index()
    out.to_csv(f"{OUT_DIR}summary_{name}.csv", index=False)
    print(f"summary_{name}.csv", out.shape)

summary_platform.csv (3, 14)
summary_campaign_type.csv (4, 13)
summary_industry.csv (5, 13)
summary_country.csv (7, 13)
summary_platform_campaign.csv (12, 12)
summary_monthly_platform.csv (36, 12)


In [ ]:
print("Lowest CAC platform:", by_platform["CAC"].idxmin(), round(by_platform["CAC"].min(), 2))
print("Highest ROAS platform:", by_platform["ROAS"].idxmax(), round(by_platform["ROAS"].max(), 2))
print("Highest conv_rate campaign type:", by_type["conv_rate"].idxmax(), round(by_type["conv_rate"].max(), 4))
print("Highest ROAS campaign type:", by_type["ROAS"].idxmax(), round(by_type["ROAS"].max(), 2))

Lowest CAC platform: TikTok Ads 21.67
Highest ROAS platform: TikTok Ads 7.62
Highest conv_rate campaign type: Search 0.0478
Highest ROAS campaign type: Search 5.31


- Industry: small spread (ROAS 4.48-5.04); Fintech weakest (20.5% of spend, 18.8% of revenue)
- Country: India best (ROAS 5.50), USA worst (4.16) with the largest spend share (15.6%)
- USA has the lowest ROAS on every platform -> country effect independent of platform
- Platform order TikTok > Meta > Google holds in all 5 industries and all 7 countries
- No clear seasonality (monthly ROAS 4.17-5.38); Google last in all 12 months, TikTok first in 11
- Conversion rate varies little across segments (4.4-4.8%)
- Funnel: losses ~95-97% at both stages; platforms differ in CTR (2.5-5.5%), not in conv. rate (4.5-4.7%)